# Exercise 3: Which orders will fail?
**Internet of Production – AI, week 1** · about 25 min · groups of 2–3

**Scenario:** You are the head of pre-production. Before an order starts, you want to know whether it is likely to fail. 11.5 % of the orders fail, and each failure wastes printer time and material.

**Goal:** After the early warning for the printer (exercise 2) you now look at the **orders**. Train classifiers on the order data, judge them with the right metrics, and give the head of pre-production three sentences of advice. Every `___` is a gap.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

DATA = Path('../../00_Data/printfarm')

## Step 1: Load the orders and define the target

In [ ]:
orders = pd.read_csv(DATA / 'Printproduction.csv', sep=';', decimal=',').dropna(how='all')
orders['Start'] = pd.to_datetime(orders['Start'], format='%d.%m.%Y %H:%M')
orders['Material qty. in g'] = pd.to_numeric(orders['Material qty. in g'].astype(str).str.replace(',', '.'), errors='coerce')
orders = orders.sort_values('Start').reset_index(drop=True)
orders['failed'] = (orders['Status'] == ___).astype(int)
print(orders.shape)
orders.head()

## Step 2: Which columns are known when an order **starts**?
Look at the columns. Select only those that the head of pre-production knows **before** the order starts. Think about each column: could you read its value at the time the order is released?

In [ ]:
orders['hour'] = orders['Start'].dt.hour
orders['weekday'] = orders['Start'].dt.dayofweek
cat = ['Printer', 'BOM Item', 'Colour']          # categorical features
num = ['Lot size', 'Target Duration', 'hour', 'weekday']   # numeric features
# TODO: check the list. Which columns of `orders` are known at order start? Remove the ones that are not.
features = cat + num
print(orders[features].dtypes)

## Step 3: Split by time
The model is used on **future** orders. Train on the first 70 % of the orders (by start time) and test on the last 30 %.

In [ ]:
cut = int(len(orders) * ___)
train, test = orders.iloc[:cut], orders.iloc[cut:]
X_train, y_train = train[features], train['failed']
X_test, y_test = test[features], test['failed']
print('train:', len(train), 'orders, failed:', y_train.sum())
print('test :', len(test), 'orders, failed:', y_test.sum())

## Step 4: Baseline
What accuracy does a model reach that **always predicts `completed`**? What are its precision and recall?

In [ ]:
def report(name, y_true, y_pred):
    print(f'{name:28s} accuracy {accuracy_score(y_true, y_pred):6.1%}   recall {recall_score(y_true, y_pred, zero_division=0):6.1%}   precision {precision_score(y_true, y_pred, zero_division=0):6.1%}')

base = DummyClassifier(strategy=___).fit(X_train, y_train)
report('baseline: always completed', y_test, base.predict(X_test))

## Step 5: Decision tree
Categorical columns are one-hot encoded. Vary the depth of the tree.

In [ ]:
def make_tree(depth):
    prep = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat)], remainder='passthrough')
    return make_pipeline(prep, DecisionTreeClassifier(max_depth=depth, class_weight='balanced', random_state=0))

for depth in [___]:        # try several depths
    m = make_tree(depth).fit(X_train, y_train)
    report(f'tree, depth {depth}', y_test, m.predict(X_test))

## Step 6 (optional, skip if time is short): KNN
KNN needs scaled numeric features.

In [ ]:
def make_knn(k):
    prep = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat), ('num', StandardScaler(), num)])
    return make_pipeline(prep, KNeighborsClassifier(n_neighbors=k))

for k in [___]:
    m = make_knn(k).fit(X_train, y_train)
    report(f'KNN, k = {k}', y_test, m.predict(X_test))

## Step 6: Confusion matrix of your best model
Choose the model you find most useful for the head of pre-production and justify your choice with a metric.

In [ ]:
best = make_tree(___).fit(X_train, y_train)      # your choice
pred = best.predict(X_test)
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=['completed', 'failed']).plot(cmap='Blues')
plt.title('Your model'); plt.show()

## Step 8: Is a simple rule just as good?
Compute the failure rate per part and test the rule *“parts with the highest failure rate fail”*.

In [ ]:
rate = orders.groupby('BOM Item')['failed'].agg(['mean', 'sum', 'size']).sort_values('mean', ascending=False)
rate['mean'] = (rate['mean'] * 100).round(1)
print(rate.head(6))

risky = list(rate.index[:___])          # how many parts?
rule = test['BOM Item'].isin(risky).astype(int)
report(f'rule: parts {risky} fail', y_test, rule)

## Step 9: What if we also use columns from *after* the order?
Run the cell. The results look excellent. **Why must you not trust them?**

In [ ]:
leaky = num + ['Actual Duration', 'Material qty. in g']
prep = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat)], remainder='passthrough')
m = make_pipeline(prep, DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=0)).fit(train[cat + leaky], y_train)
report('tree WITH Actual Duration, qty', y_test, m.predict(test[cat + leaky]))
print()
print(orders.groupby('Status')[['Actual Duration', 'Material qty. in g']].median())

**Why is this result not usable?**

*Your answer:*

## Step 10: Why do orders fail? Link the orders to the disturbances
A printer disturbance is logged with its time stamp. Check for every order whether a disturbance of **its printer** occurred between the start and the end of the order.

In [ ]:
def load_errors(k):
    e = pd.read_csv(DATA / f'Printer_{k}_error.csv', sep=';').dropna(how='all')
    e['Timestamp'] = pd.to_datetime(e['Timestamp'], format='%d.%m.%Y %H:%M')
    e['Printer'] = k
    return e

errors = pd.concat([load_errors(k) for k in (0, 1, 2)])
orders['End'] = orders['Start'] + pd.to_timedelta(orders['Actual Duration'], unit='m')

def disturbance(o):
    tol = pd.Timedelta(minutes=5)
    e = errors[(errors['Printer'] == o['Printer']) & (errors['Timestamp'] >= o['Start'] - tol) & (errors['Timestamp'] <= o['End'] + tol)]
    return e['Error'].iloc[0] if len(e) else 'none'

orders['disturbance'] = orders.apply(disturbance, axis=1)
print(pd.crosstab(orders[___], orders['disturbance'] != 'none', colnames=['disturbance during the order']))
print()
print(pd.crosstab(orders.loc[orders['failed'] == 1, 'Printer'], orders.loc[orders['failed'] == 1, 'disturbance']))

**What does this mean for the prediction of failed orders, and how does it connect to exercise 2?**

*Your answer:*

## Your advice (three sentences)
What should the head of pre-production do? Which model or rule, which metric, and what is still missing?

*Your answer:*